# 07 — Model 3: Support Vector Machine (SVM)


> **Notebook 7 of 11** — part of the *Heart Disease Detection using Explainable AI* project.
> Run the notebooks **in order**, from 01 to 11.

---

## 🎯 How an SVM works, explained simply

An SVM looks for the boundary with the **widest possible safety margin** between the two groups.
Rather than any old dividing line, it finds the one that stays as far as possible from the nearest
patients on both sides.

The **RBF kernel** lets that boundary be *curved* instead of straight. It does this with a clever
trick: imagine lifting the patients into a higher-dimensional space where a curve becomes a flat
plane, drawing the plane there, then dropping everything back down.

| Strength | Weakness |
|---|---|
| Powerful, elegant, handles curved boundaries | **Slow** on large datasets |
| Very effective in high dimensions | Gives a score, not a probability, by default |

## ⚙️ The settings we tune

* **`C`** — how much we punish mistakes. High `C` = a wigglier boundary fitting training data
  closely; low `C` = a smoother, more general boundary.
* **`gamma`** — how far a single patient's influence reaches. High gamma = each patient only
  affects its immediate neighbourhood.

## 💡 An engineering decision worth defending in your viva

An RBF SVM's training time grows roughly with the **square** of the number of patients. On 54,000
patients that becomes impractical — hours, not minutes.

So we train it on a **random sample of 15,000 patients**. This is a normal, documented trade-off,
not laziness:

* The sample is drawn randomly from a dataset that is already balanced 50/50, so it stays fair.
* You will see below that the accuracy comes out essentially identical to the other two models.

Say this out loud in your demo. Knowing *why* you sampled, and proving it cost you nothing, is
better than pretending the problem does not exist.

In [ ]:
import os, json, time, warnings
import numpy as np, pandas as pd, joblib
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, roc_curve, classification_report)

warnings.filterwarnings("ignore"); sns.set_style("whitegrid")
RANDOM_STATE = 42; np.random.seed(RANDOM_STATE)
DATA, MODELS = "../data", "../models"

# --- load what notebook 04 prepared ---
prep = np.load(f"{DATA}/prepared.npz", allow_pickle=True)
FEATURES = list(prep["features"])
X_train, X_test = prep["X_train"], prep["X_test"]
y_train, y_test = prep["y_train"], prep["y_test"]
scaler = joblib.load(f"{MODELS}/scaler.pkl")

X_train_scaled = scaler.transform(X_train)
X_test_scaled  = scaler.transform(X_test)
cv_plan = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print(f"Training patients : {len(X_train):,}")
print(f"Test patients     : {len(X_test):,}")
print(f"Features          : {FEATURES}")

from sklearn.svm import SVC

In [ ]:
def save_results(model_name, model, best_params, cv_scores, seconds):
    """Grade the model on the SEALED test set and append the scores to models/metrics.json."""
    y_pred  = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    keep = max(1, len(fpr) // 200)

    entry = {
        "accuracy":  float(accuracy_score(y_test, y_pred)),
        "precision": float(precision_score(y_test, y_pred)),
        "recall":    float(recall_score(y_test, y_pred)),
        "f1":        float(f1_score(y_test, y_pred)),
        "roc_auc":   float(roc_auc_score(y_test, y_proba)),
        "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
        "fpr": fpr[::keep].tolist(), "tpr": tpr[::keep].tolist(),
        "cv_scores": [float(s) for s in cv_scores],
        "cv_mean": float(np.mean(cv_scores)), "cv_std": float(np.std(cv_scores)),
        "best_params": {k: str(v) for k, v in best_params.items()},
        "train_seconds": round(seconds, 1),
    }

    path = f"{MODELS}/metrics.json"
    all_metrics = json.load(open(path)) if os.path.exists(path) else {}
    all_metrics[model_name] = entry
    json.dump(all_metrics, open(path, "w"), indent=2)

    print(f"\n{'=' * 58}\n  {model_name}\n{'=' * 58}")
    print(f"  Accuracy   : {entry['accuracy']:.4f}   (out of 100 patients, "
          f"{entry['accuracy']*100:.0f} labelled correctly)")
    print(f"  Precision  : {entry['precision']:.4f}   (when we say 'disease', we are right this often)")
    print(f"  Recall     : {entry['recall']:.4f}   (of all truly sick, we caught this many)")
    print(f"  F1 Score   : {entry['f1']:.4f}   (balance of precision and recall)")
    print(f"  ROC-AUC    : {entry['roc_auc']:.4f}   (0.5 = coin toss, 1.0 = perfect)")
    print(f"  CV accuracy: {entry['cv_mean']:.4f} +/- {entry['cv_std']:.4f}")
    return y_proba

print("Helper ready.")

## 1. Take a fair random sample

In [ ]:
sample_idx = np.random.RandomState(RANDOM_STATE).choice(
    len(X_train_scaled), size=15000, replace=False)
X_svm, y_svm = X_train_scaled[sample_idx], y_train[sample_idx]

print(f"Full training set   : {len(X_train_scaled):,} patients")
print(f"SVM training sample : {len(X_svm):,} patients")
print(f"\nDisease rate in the full set : {y_train.mean()*100:.2f}%")
print(f"Disease rate in the sample   : {y_svm.mean()*100:.2f}%")
print("\nThe rates match, so the sample is representative and the trade-off is fair.")

## 2. Hyperparameter tuning

In [ ]:
print("Tuning SVM... (about 1-2 minutes)")
start = time.time()

# We tune on an even smaller slice, because grid search trains many SVMs
grid = GridSearchCV(
    estimator=SVC(kernel="rbf", random_state=RANDOM_STATE),
    param_grid={"C": [1.0, 5.0], "gamma": ["scale", 0.05]},
    cv=3, scoring="roc_auc", n_jobs=-1)
grid.fit(X_svm[:6000], y_svm[:6000])

print(f"\nBest settings   : {grid.best_params_}")
print(f"Best CV ROC-AUC : {grid.best_score_:.4f}")
print(f"Took {time.time()-start:.0f} seconds")

res = pd.DataFrame(grid.cv_results_)
param_cols = [c for c in res.columns if c.startswith("param_")]
res[param_cols + ["mean_test_score"]].sort_values("mean_test_score",
                                                  ascending=False).round(4)

## 3. Cross-validation

In [ ]:
base_model = SVC(kernel="rbf", random_state=RANDOM_STATE, **grid.best_params_)
cv_scores = cross_val_score(base_model, X_svm, y_svm, cv=cv_plan,
                            scoring="accuracy", n_jobs=-1)

print("Accuracy on each of the 5 folds:", np.round(cv_scores, 4))
print(f"Average   : {cv_scores.mean():.4f}")
print(f"Variation : +/- {cv_scores.std():.4f}")

## Why `CalibratedClassifierCV`?

A model's raw output is a **score**, not necessarily an honest probability. A model might output
0.80 for a group of patients of whom only 65% are really sick — the *ranking* is right but the
*number* is exaggerated, and different algorithms exaggerate differently.

That matters for us, because the website shows all three percentages **side by side**. If one says
45% and another says 78%, the user cannot tell whom to believe.

**Isotonic calibration** learns a correction curve on held-out folds and re-maps every score so
that among patients given 70%, roughly 70% really are sick. Notebook 08 measures how well this
worked.


For the SVM there is a second, more basic reason we need this. A plain `SVC` has **no
`predict_proba` at all** — it only outputs a raw distance from the boundary. Calibration is what
turns that distance into a real probability we can display on the website.

In [ ]:
final_model = CalibratedClassifierCV(base_model, cv=3, method="isotonic")
final_model.fit(X_svm, y_svm)

proba = save_results("SVM", final_model, grid.best_params_, cv_scores, time.time() - start)

In [ ]:
print(classification_report(y_test, final_model.predict(X_test_scaled),
                            target_names=["Healthy", "Heart disease"], digits=4))

cm = confusion_matrix(y_test, final_model.predict(X_test_scaled))
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt=",d", cmap="Oranges", cbar=False, ax=ax,
            xticklabels=["Predicted\nHealthy", "Predicted\nDisease"],
            yticklabels=["Actually\nHealthy", "Actually\nDisease"])
ax.set_title("SVM — confusion matrix")
plt.tight_layout(); plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"This model is CAUTIOUS: it raises fewer false alarms ({fp:,})")
print(f"but misses more sick patients ({fn:,}) than the other two.")
print("That is a precision-over-recall trade-off, which notebook 08 examines.")

In [ ]:
joblib.dump(final_model, f"{MODELS}/svm.pkl")
np.save(f"{MODELS}/proba__svm.npy", proba)
print(f"Saved {MODELS}/svm.pkl")

---
## ✅ Summary

* Trained on a fair **15,000-patient random sample** — a documented engineering trade-off.
* Tuned `C` and `gamma`, validated with 5-fold cross-validation.
* Calibration was **essential** here, because a plain SVC has no probabilities at all.
* Accuracy around **73%**, with the highest precision but the lowest recall of the three.

### ▶️ Next: `08_Model_Comparison.ipynb` — put all three head to head.